<a href="https://colab.research.google.com/github/Poorvi-M/ParkinSense-AI/blob/main/Parkinsons.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Parkinsons Early Detection**

In [ ]:
!pip install praat-parselmouth


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

dataset_path = "/content/drive/MyDrive/Italian Parkinson's Voice and speech"

for group in os.listdir(dataset_path):
    folder = os.path.join(dataset_path, group)
    count = sum(len(files) for _, _, files in os.walk(folder))
    print(f"{group}: {count} files")

In [ ]:
import os

dataset_path = "/content/drive/MyDrive/Italian Parkinson's Voice and speech"

pd_path = os.path.join(dataset_path, "28 People with Parkinson's disease")
hc_path = os.path.join(dataset_path, "22 Elderly Healthy Control")

print("PD subfolders:", os.listdir(pd_path))
print("HC subfolders:", os.listdir(hc_path))

In [ ]:
# Check one PD subfolder
pd_sub = os.path.join(pd_path, '1-5')
print("Inside 1-5:", os.listdir(pd_sub))
# Check one HC subfolder
hc_sub = os.path.join(hc_path, 'TERESA M')
print("Inside TERESA M:", os.listdir(hc_sub))

In [ ]:
import os
import numpy as np
import librosa
import librosa.display
import matplotlib.pyplot as plt
from pathlib import Path
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
import torch.nn as nn
from sklearn.model_selection import GroupShuffleSplit
from PIL import Image
import io

In [ ]:
dataset_path = "/content/drive/MyDrive/Italian Parkinson's Voice and speech"
pd_path = os.path.join(dataset_path, "28 People with Parkinson's disease")
hc_path = os.path.join(dataset_path, "22 Elderly Healthy Control")

# Recording types to keep
KEEP = ('VA1','VA2','VE1','VE2','VI1','VI2','VO1','VO2','VU1','VU2','D1','D2')

def collect_files(group_path, label):
    records = []
    for root, dirs, files in os.walk(group_path):
        # skip excel files
        for f in files:
            if f.endswith('.wav') and f.startswith(KEEP):
                patient_id = os.path.basename(root)
                records.append({
                    'path': os.path.join(root, f),
                    'label': label,
                    'patient': f"{label}_{patient_id}"
                })
    return records

pd_files  = collect_files(pd_path, label=1)
hc_files  = collect_files(hc_path, label=0)
all_files = pd_files + hc_files

print(f"PD files:  {len(pd_files)}")
print(f"HC files:  {len(hc_files)}")
print(f"Total:     {len(all_files)}")

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

paths    = [r['path']    for r in all_files]
labels   = [r['label']   for r in all_files]
patients = [r['patient'] for r in all_files]

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(paths, labels, groups=patients))

train_records = [all_files[i] for i in train_idx]
test_records  = [all_files[i] for i in test_idx]

print(f"Train: {len(train_records)} files")
print(f"Test:  {len(test_records)} files")

In [ ]:
def audio_to_melspec(file_path, sr=22050, n_mels=128, duration=3):
    audio, _ = librosa.load(file_path, sr=sr, duration=duration)

    # pad if shorter than duration
    target_len = sr * duration
    if len(audio) < target_len:
        audio = np.pad(audio, (0, target_len - len(audio)))

    mel = librosa.feature.melspectrogram(y=audio, sr=sr, n_mels=n_mels)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    return mel_db

In [ ]:
class ParkinsonDataset(Dataset):
    def __init__(self, records, transform=None):
        self.records = records
        self.transform = transform

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        record = self.records[idx]

        # audio → mel spectrogram → PIL image
        mel = audio_to_melspec(record['path'])

        # normalize to 0-255 and convert to RGB
        mel = ((mel - mel.min()) / (mel.max() - mel.min()) * 255).astype(np.uint8)
        img = Image.fromarray(mel).convert('RGB')

        if self.transform:
            img = self.transform(img)

        return img, record['label']

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.2)        # moved after ToTensor
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

train_dataset = ParkinsonDataset(train_records, transform=train_transform)
test_dataset  = ParkinsonDataset(test_records,  transform=test_transform)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=16, shuffle=False)

print(f"Train batches: {len(train_loader)}")
print(f"Test batches:  {len(test_loader)}")

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using: {device}")

model = models.resnet18(weights='IMAGENET1K_V1')

# freeze all layers except the final one
for param in model.parameters():
    param.requires_grad = False

# replace final layer with 2-class output
model.fc = nn.Linear(model.fc.in_features, 2)
model = model.to(device)

In [ ]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)

def train(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, correct = 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        correct += (outputs.argmax(1) == labels).sum().item()
    return total_loss/len(loader), correct/len(loader.dataset)

In [ ]:
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct = 0, 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            total_loss += loss.item()
            correct += (outputs.argmax(1) == labels).sum().item()
            all_preds.extend(outputs.argmax(1).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    return total_loss/len(loader), correct/len(loader.dataset), all_preds, all_labels

In [ ]:
EPOCHS = 20

for epoch in range(EPOCHS):
    train_loss, train_acc = train(model, train_loader, optimizer, criterion, device)
    val_loss, val_acc, _, _ = evaluate(model, test_loader, criterion, device)
    print(f"Epoch {epoch+1}/{EPOCHS} | "
          f"Train Loss: {train_loss:.3f} Acc: {train_acc:.3f} | "
          f"Val Loss: {val_loss:.3f} Acc: {val_acc:.3f}")

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

_, _, preds, true_labels = evaluate(model, test_loader, criterion, device)

print(classification_report(true_labels, preds, target_names=['HC', 'PD']))

cm = confusion_matrix(true_labels, preds)
sns.heatmap(cm, annot=True, fmt='d', xticklabels=['HC','PD'], yticklabels=['HC','PD'])
plt.title('Confusion Matrix')
plt.show()

# save model
torch.save(model.state_dict(), '/content/drive/MyDrive/parkinsons_resnet18.pt')
print("Model saved.")

In [ ]:
# unfreeze layer4 and fc
for name, param in model.named_parameters():
    if 'layer4' in name or 'fc' in name:
        param.requires_grad = True

# lower learning rate for fine-tuning
optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=0.0001
)

EPOCHS = 15

for epoch in range(EPOCHS):
    train_loss, train_acc = train(model, train_loader, optimizer, criterion, device)
    val_loss, val_acc, _, _ = evaluate(model, test_loader, criterion, device)
    print(f"Epoch {epoch+1}/{EPOCHS} | "
          f"Train Loss: {train_loss:.3f} Acc: {train_acc:.3f} | "
          f"Val Loss: {val_loss:.3f} Acc: {val_acc:.3f}")

# evaluate and save
_, _, preds, true_labels = evaluate(model, test_loader, criterion, device)
print(classification_report(true_labels, preds, target_names=['HC', 'PD']))
torch.save(model.state_dict(), '/content/drive/MyDrive/parkinsons_resnet18_finetuned.pt')
print("Fine-tuned model saved.")

In [ ]:
def calibrated_severity(file_path):
    try:
        sound = parselmouth.Sound(file_path)
        point_process = call(sound, "To PointProcess (periodic, cc)", 75, 500)

        jitter  = call(point_process, "Get jitter (local)", 0, 0, 0.0001, 0.02, 1.3)
        shimmer = call([sound, point_process], "Get shimmer (local)", 0, 0, 0.0001, 0.02, 1.3, 1.6)

        # add HNR — lower HNR = more noise = more severe
        harmonicity = call(sound, "To Harmonicity (cc)", 0.01, 75, 0.1, 1.0)
        hnr = call(harmonicity, "Get mean", 0, 0)

        return jitter, shimmer, hnr
    except:
        return None, None, None

# compute across all files
raw_scores = []
for r in all_files:
    j, s, h = calibrated_severity(r['path'])
    if j is not None:
        raw_scores.append({'path': r['path'], 'label': r['label'],
                          'jitter': j, 'shimmer': s, 'hnr': h})

# compute normalization stats
jitter_arr  = np.array([x['jitter']  for x in raw_scores])
shimmer_arr = np.array([x['shimmer'] for x in raw_scores])
hnr_arr     = np.array([x['hnr']     for x in raw_scores])

j_mean, j_std = jitter_arr.mean(),  jitter_arr.std()
s_mean, s_std = shimmer_arr.mean(), shimmer_arr.std()
h_mean, h_std = hnr_arr.mean(),     hnr_arr.std()

def get_severity_score(file_path):
    j, s, h = calibrated_severity(file_path)
    if j is None:
        return None
    j_z =  (j - j_mean) / j_std   # higher jitter = worse
    s_z =  (s - s_mean) / s_std   # higher shimmer = worse
    h_z = -(h - h_mean) / h_std   # lower HNR = worse (inverted)
    raw = (j_z + s_z + h_z) / 3
    return round(np.clip((raw + 3) / 6 * 10, 0, 10), 2)

# verify
pd_scores = [get_severity_score(r['path']) for r in pd_files[:20]]
hc_scores = [get_severity_score(r['path']) for r in hc_files[:20]]
pd_scores = [s for s in pd_scores if s is not None]
hc_scores = [s for s in hc_scores if s is not None]

print(f"Avg PD severity: {np.mean(pd_scores):.2f}/10")
print(f"Avg HC severity: {np.mean(hc_scores):.2f}/10")
print(f"Difference:      {np.mean(pd_scores) - np.mean(hc_scores):.2f}")

In [ ]:
!pip install grad-cam

In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
import cv2

def visualize_gradcam(model, dataset, records, idx, device):
    model.eval()

    img_tensor, label = dataset[idx]
    target_layers = [model.layer4[-1]]

    cam = GradCAM(model=model, target_layers=target_layers)

    input_tensor = img_tensor.unsqueeze(0).to(device)
    grayscale_cam = cam(input_tensor=input_tensor)[0]

    mel = audio_to_melspec(records[idx]['path'])
    mel_norm = ((mel - mel.min()) / (mel.max() - mel.min())).astype(np.float32)
    mel_rgb = cv2.applyColorMap((mel_norm * 255).astype(np.uint8), cv2.COLORMAP_MAGMA)
    mel_rgb = cv2.cvtColor(mel_rgb, cv2.COLOR_BGR2RGB) / 255.0
    mel_rgb = cv2.resize(mel_rgb, (224, 224))

    visualization = show_cam_on_image(mel_rgb, grayscale_cam, use_rgb=True)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].imshow(mel_rgb)
    axes[0].set_title(f"Mel Spectrogram | True: {'PD' if label==1 else 'HC'}")
    axes[0].set_xlabel("Time")
    axes[0].set_ylabel("Mel Frequency")

    axes[1].imshow(visualization)
    axes[1].set_title("Grad-CAM Overlay")
    axes[1].set_xlabel("Time")
    axes[1].set_ylabel("Mel Frequency")

    plt.tight_layout()
    plt.savefig('/content/drive/MyDrive/gradcam_fixed.png', dpi=150)
    plt.show()
    print(f"True label: {'PD' if label==1 else 'HC'}")

visualize_gradcam(model, test_dataset, test_records, 0, device)

In [ ]:
import parselmouth
from parselmouth.praat import call
import numpy as np

def compute_all_severity_scores(records):
    scores = []
    for r in records:
        try:
            sound = parselmouth.Sound(r['path'])
            point_process = call(sound, "To PointProcess (periodic, cc)", 75, 500)
            jitter = call(point_process, "Get jitter (local)", 0, 0, 0.0001, 0.02, 1.3)
            shimmer = call([sound, point_process], "Get shimmer (local)", 0, 0, 0.0001, 0.02, 1.3, 1.6)
            scores.append({'path': r['path'], 'label': r['label'], 'jitter': jitter, 'shimmer': shimmer})
        except:
            scores.append({'path': r['path'], 'label': r['label'], 'jitter': None, 'shimmer': None})
    return scores

all_scores   = compute_all_severity_scores(all_files)
valid        = [s for s in all_scores if s['jitter'] is not None]
jitter_vals  = np.array([s['jitter']  for s in valid])
shimmer_vals = np.array([s['shimmer'] for s in valid])
jitter_mean, jitter_std   = jitter_vals.mean(), jitter_vals.std()
shimmer_mean, shimmer_std = shimmer_vals.mean(), shimmer_vals.std()

def calibrated_severity(file_path):
    try:
        sound = parselmouth.Sound(file_path)
        point_process = call(sound, "To PointProcess (periodic, cc)", 75, 500)
        jitter  = call(point_process, "Get jitter (local)", 0, 0, 0.0001, 0.02, 1.3)
        shimmer = call([sound, point_process], "Get shimmer (local)", 0, 0, 0.0001, 0.02, 1.3, 1.6)
        j_z   = (jitter  - jitter_mean)  / jitter_std
        s_z   = (shimmer - shimmer_mean) / shimmer_std
        raw   = (j_z + s_z) / 2
        score = round(np.clip((raw + 3) / 6 * 10, 0, 10), 2)
        return score
    except:
        return None

pd_scores = [calibrated_severity(r['path']) for r in pd_files[:20]]
hc_scores = [calibrated_severity(r['path']) for r in hc_files[:20]]
pd_scores = [s for s in pd_scores if s is not None]
hc_scores = [s for s in hc_scores if s is not None]

print(f"Avg PD severity: {np.mean(pd_scores):.2f}/10")
print(f"Avg HC severity: {np.mean(hc_scores):.2f}/10")

In [ ]:
# compare raw feature values between groups before any normalization
pd_raw  = [r for r in raw_scores if r['label'] == 1]
hc_raw  = [r for r in raw_scores if r['label'] == 0]

pd_jitter  = np.mean([r['jitter']  for r in pd_raw])
hc_jitter  = np.mean([r['jitter']  for r in hc_raw])

pd_shimmer = np.mean([r['shimmer'] for r in pd_raw])
hc_shimmer = np.mean([r['shimmer'] for r in hc_raw])

pd_hnr     = np.mean([r['hnr']     for r in pd_raw])
hc_hnr     = np.mean([r['hnr']     for r in hc_raw])

print(f"{'Feature':<12} {'PD mean':>10} {'HC mean':>10} {'Difference':>12}")
print("-" * 46)
print(f"{'Jitter':<12} {pd_jitter:>10.5f} {hc_jitter:>10.5f} {pd_jitter-hc_jitter:>12.5f}")
print(f"{'Shimmer':<12} {pd_shimmer:>10.5f} {hc_shimmer:>10.5f} {pd_shimmer-hc_shimmer:>12.5f}")
print(f"{'HNR':<12} {pd_hnr:>10.3f} {hc_hnr:>10.3f} {pd_hnr-hc_hnr:>12.3f}")

In [ ]:
# check which recording types are being processed
from collections import Counter

def get_recording_type(path):
    filename = os.path.basename(path)
    return filename[:3]  # first 3 chars = recording code

pd_types = Counter([get_recording_type(r['path']) for r in pd_raw])
hc_types = Counter([get_recording_type(r['path']) for r in hc_raw])

print("PD recording types:", dict(pd_types))
print("HC recording types:", dict(hc_types))


In [ ]:
VOWEL_CODES = ('VA1', 'VA2', 'VE1', 'VE2', 'VI1', 'VI2', 'VO1', 'VO2', 'VU1', 'VU2')

# recompute using vowels only
vowel_files = [r for r in all_files if os.path.basename(r['path']).startswith(VOWEL_CODES)]

print(f"Total vowel files: {len(vowel_files)}")
print(f"PD vowel files: {sum(1 for r in vowel_files if r['label']==1)}")
print(f"HC vowel files: {sum(1 for r in vowel_files if r['label']==0)}")

# recompute raw scores on vowels only
raw_scores = []
for r in vowel_files:
    j, s, h = calibrated_severity(r['path'])
    if j is not None:
        raw_scores.append({'path': r['path'], 'label': r['label'],
                          'jitter': j, 'shimmer': s, 'hnr': h})

pd_raw = [r for r in raw_scores if r['label'] == 1]
hc_raw = [r for r in raw_scores if r['label'] == 0]

pd_jitter  = np.mean([r['jitter']  for r in pd_raw])
hc_jitter  = np.mean([r['jitter']  for r in hc_raw])
pd_shimmer = np.mean([r['shimmer'] for r in pd_raw])
hc_shimmer = np.mean([r['shimmer'] for r in hc_raw])
pd_hnr     = np.mean([r['hnr']     for r in pd_raw])
hc_hnr     = np.mean([r['hnr']     for r in hc_raw])

print(f"\n{'Feature':<12} {'PD mean':>10} {'HC mean':>10} {'Difference':>12}")
print("-" * 46)
print(f"{'Jitter':<12} {pd_jitter:>10.5f} {hc_jitter:>10.5f} {pd_jitter-hc_jitter:>12.5f}")
print(f"{'Shimmer':<12} {pd_shimmer:>10.5f} {hc_shimmer:>10.5f} {pd_shimmer-hc_shimmer:>12.5f}")
print(f"{'HNR':<12} {pd_hnr:>10.3f} {hc_hnr:>10.3f} {pd_hnr-hc_hnr:>12.3f}")

In [ ]:
# test on one vowel file directly
test_path = vowel_files[0]['path']
print("Testing file:", test_path)

try:
    sound = parselmouth.Sound(test_path)
    print("Sound loaded OK")

    point_process = call(sound, "To PointProcess (periodic, cc)", 75, 500)
    print("PointProcess OK")

    jitter = call(point_process, "Get jitter (local)", 0, 0, 0.0001, 0.02, 1.3)
    print(f"Jitter: {jitter}")

    shimmer = call([sound, point_process], "Get shimmer (local)", 0, 0, 0.0001, 0.02, 1.3, 1.6)
    print(f"Shimmer: {shimmer}")

    harmonicity = call(sound, "To Harmonicity (cc)", 0.01, 75, 0.1, 1.0)
    hnr = call(harmonicity, "Get mean", 0, 0)
    print(f"HNR: {hnr}")

except Exception as e:
    print(f"Error: {e}")

In [ ]:
VOWEL_CODES = ('VA1', 'VA2', 'VE1', 'VE2', 'VI1', 'VI2', 'VO1', 'VO2', 'VU1', 'VU2')
vowel_files = [r for r in all_files if os.path.basename(r['path']).startswith(VOWEL_CODES)]

# step 1 — collect raw features
raw_scores = []
for r in vowel_files:
    try:
        sound = parselmouth.Sound(r['path'])
        point_process = call(sound, "To PointProcess (periodic, cc)", 75, 500)
        jitter  = call(point_process, "Get jitter (local)", 0, 0, 0.0001, 0.02, 1.3)
        shimmer = call([sound, point_process], "Get shimmer (local)", 0, 0, 0.0001, 0.02, 1.3, 1.6)
        harmonicity = call(sound, "To Harmonicity (cc)", 0.01, 75, 0.1, 1.0)
        hnr = call(harmonicity, "Get mean", 0, 0)
        raw_scores.append({'path': r['path'], 'label': r['label'],
                           'jitter': jitter, 'shimmer': shimmer, 'hnr': hnr})
    except:
        pass

print(f"Successfully processed: {len(raw_scores)} / {len(vowel_files)} files")

# step 2 — compute normalization stats
pd_raw = [r for r in raw_scores if r['label'] == 1]
hc_raw = [r for r in raw_scores if r['label'] == 0]

jitter_arr  = np.array([r['jitter']  for r in raw_scores])
shimmer_arr = np.array([r['shimmer'] for r in raw_scores])
hnr_arr     = np.array([r['hnr']     for r in raw_scores])

j_mean, j_std = jitter_arr.mean(), jitter_arr.std()
s_mean, s_std = shimmer_arr.mean(), shimmer_arr.std()
h_mean, h_std = hnr_arr.mean(), hnr_arr.std()

# step 3 — print raw group differences
pd_jitter  = np.mean([r['jitter']  for r in pd_raw])
hc_jitter  = np.mean([r['jitter']  for r in hc_raw])
pd_shimmer = np.mean([r['shimmer'] for r in pd_raw])
hc_shimmer = np.mean([r['shimmer'] for r in hc_raw])
pd_hnr     = np.mean([r['hnr']     for r in pd_raw])
hc_hnr     = np.mean([r['hnr']     for r in hc_raw])

print(f"\n{'Feature':<12} {'PD mean':>10} {'HC mean':>10} {'Difference':>12}")
print("-" * 46)
print(f"{'Jitter':<12} {pd_jitter:>10.5f} {hc_jitter:>10.5f} {pd_jitter-hc_jitter:>12.5f}")
print(f"{'Shimmer':<12} {pd_shimmer:>10.5f} {hc_shimmer:>10.5f} {pd_shimmer-hc_shimmer:>12.5f}")
print(f"{'HNR':<12} {pd_hnr:>10.3f} {hc_hnr:>10.3f} {pd_hnr-hc_hnr:>12.3f}")

In [ ]:
def get_cnn_severity(model, dataset, idx, device):
    model.eval()
    img_tensor, true_label = dataset[idx]
    input_tensor = img_tensor.unsqueeze(0).to(device)

    with torch.no_grad():
        output = model(input_tensor)
        prob = torch.softmax(output, dim=1)[0][1].item()  # probability of PD

    # map probability to severity score
    if prob < 0.5:
        severity = round(prob * 6, 2)        # 0-3 range for likely HC
    elif prob < 0.75:
        severity = round(3 + (prob - 0.5) * 16, 2)   # 3-7 range
    else:
        severity = round(7 + (prob - 0.75) * 12, 2)  # 7-10 range

    severity = round(min(max(severity, 0), 10), 2)
    return prob, severity

# test on several samples
print(f"{'Index':<8} {'True Label':<12} {'PD Probability':<16} {'Severity Score'}")
print("-" * 52)
for i in range(10):
    img_tensor, true_label = test_dataset[i]
    prob, severity = get_cnn_severity(model, test_dataset, i, device)
    print(f"{i:<8} {'PD' if true_label==1 else 'HC':<12} {prob:<16.3f} {severity}/10")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

features = {
    'Jitter': ([r['jitter'] for r in pd_raw], [r['jitter'] for r in hc_raw]),
    'Shimmer': ([r['shimmer'] for r in pd_raw], [r['shimmer'] for r in hc_raw]),
    'HNR': ([r['hnr'] for r in pd_raw], [r['hnr'] for r in hc_raw])
}

for ax, (feature, (pd_vals, hc_vals)) in zip(axes, features.items()):
    ax.boxplot([pd_vals, hc_vals], labels=['PD', 'HC'])
    ax.set_title(f'{feature} Distribution')
    ax.set_ylabel(feature)

plt.suptitle('Acoustic Features: PD vs HC\n(Elderly Italian Dataset)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/acoustic_features_analysis.png', dpi=150)
plt.show()
print("Saved to Drive.")

In [ ]:
print(f"{'Index':<8} {'True Label':<12} {'PD Probability':<16} {'Severity Score'}")
print("-" * 52)
for i in range(len(test_records)):
    prob, severity = get_cnn_severity(model, test_dataset, i, device)
    true_label = test_records[i]['label']
    print(f"{i:<8} {'PD' if true_label==1 else 'HC':<12} {prob:<16.3f} {severity}/10")

In [ ]:
def get_cnn_severity(model, dataset, idx, device, temperature=3.0):
    model.eval()
    img_tensor, true_label = dataset[idx]
    input_tensor = img_tensor.unsqueeze(0).to(device)

    with torch.no_grad():
        output = model(input_tensor)
        # temperature scaling softens the probabilities
        scaled_output = output / temperature
        prob = torch.softmax(scaled_output, dim=1)[0][1].item()

    if prob < 0.5:
        severity = round(prob * 6, 2)
    elif prob < 0.75:
        severity = round(3 + (prob - 0.5) * 16, 2)
    else:
        severity = round(7 + (prob - 0.75) * 12, 2)

    severity = round(min(max(severity, 0), 10), 2)
    return prob, severity

# test with temperature scaling
print(f"{'Index':<8} {'True Label':<12} {'PD Probability':<16} {'Severity Score'}")
print("-" * 52)
for i in range(len(test_records)):
    prob, severity = get_cnn_severity(model, test_dataset, i, device)
    true_label = test_records[i]['label']
    print(f"{i:<8} {'PD' if true_label==1 else 'HC':<12} {prob:<16.3f} {severity}/10")

Dataset:     Italian Parkinson's Voice & Speech (65 subjects)
Model:       ResNet18 fine-tuned (transfer learning)
Accuracy:    83%
PD Recall:   85%
Finding:     Traditional acoustic features (jitter, shimmer, HNR)
             insufficient for age-matched PD detection
Contribution: CNN-based mel spectrogram approach outperforms
             acoustic feature methods